# Hirsch Cultural Literacy Comparison

Compare the Jeopardy-derived concept rankings against E.D. Hirsch's *New Dictionary of Cultural Literacy* (~6,900 entries).

Questions:
1. How many Hirsch entries appear in our Jeopardy concept list?
2. Which Hirsch entries are most/least represented on Jeopardy?
3. Which high-importance Jeopardy concepts are NOT on Hirsch's list?
4. How does coverage vary by Hirsch section and Jeopardy entity type?

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load both datasets
jeopardy = pd.read_csv("../data/jeopardy_concepts_v3.csv", index_col=0, engine="python", on_bad_lines="skip")
hirsch = pd.read_csv("../data/hirsch_cultural_literacy_terms.csv")

print(f"Jeopardy concepts: {len(jeopardy):,}")
print(f"Hirsch terms: {len(hirsch):,}")
print(f"\nHirsch sections: {hirsch['section'].nunique()}")
print(hirsch['section'].value_counts().to_string())

## 1. Normalize and Match

Normalize both lists to lowercase, strip articles/punctuation, then do exact and fuzzy matching.

In [ ]:
def normalize(text):
    """Normalize a term for matching."""
    if not isinstance(text, str):
        return ""
    s = text.lower().strip()
    # Remove leading articles
    s = re.sub(r"^(the|a|an)\s+", "", s)
    # Remove punctuation except hyphens and periods in abbreviations
    s = re.sub(r"[,;:!?'\"()\[\]]", "", s)
    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Normalize Hirsch terms
hirsch["norm"] = hirsch["term"].apply(normalize)

# For Hirsch entries in "Last, First" format, also create "First Last" variant
def flip_name(term):
    """Convert 'Last, First' to 'first last' for matching."""
    if ", " in term:
        parts = term.split(", ", 1)
        if len(parts) == 2 and parts[1][0:1].isupper():
            return normalize(parts[1] + " " + parts[0])
    return None

hirsch["norm_flipped"] = hirsch["term"].apply(flip_name)

# Build lookup sets from Hirsch
hirsch_norms = set(hirsch["norm"].dropna())
hirsch_flipped = set(hirsch["norm_flipped"].dropna())
hirsch_all = hirsch_norms | hirsch_flipped

# Normalize Jeopardy concepts
jeopardy["norm"] = jeopardy["concept"].apply(normalize)

# Also try matching display_name
jeopardy["norm_display"] = jeopardy["display_name"].apply(normalize)

print(f"Hirsch normalized terms: {len(hirsch_norms):,}")
print(f"Hirsch flipped names: {len(hirsch_flipped):,}")
print(f"Hirsch total lookup: {len(hirsch_all):,}")
print(f"\nJeopardy normalized: {len(jeopardy):,}")

# Sample
print(f"\nSample Hirsch norms: {sorted(list(hirsch_norms))[:20]}")
print(f"Sample Hirsch flipped: {sorted(list(hirsch_flipped))[:20]}")

In [ ]:
# Exact matching
jeopardy["in_hirsch"] = (
    jeopardy["norm"].isin(hirsch_all) | 
    jeopardy["norm_display"].isin(hirsch_all)
)

# Also check: which Hirsch terms appear in Jeopardy?
jeopardy_norms = set(jeopardy["norm"].dropna()) | set(jeopardy["norm_display"].dropna())
hirsch["in_jeopardy"] = (
    hirsch["norm"].isin(jeopardy_norms) | 
    hirsch["norm_flipped"].isin(jeopardy_norms)
)

n_jeopardy_in_hirsch = jeopardy["in_hirsch"].sum()
n_hirsch_in_jeopardy = hirsch["in_jeopardy"].sum()

print(f"Jeopardy concepts found in Hirsch: {n_jeopardy_in_hirsch:,} / {len(jeopardy):,} ({100*n_jeopardy_in_hirsch/len(jeopardy):.1f}%)")
print(f"Hirsch terms found in Jeopardy: {n_hirsch_in_jeopardy:,} / {len(hirsch):,} ({100*n_hirsch_in_jeopardy/len(hirsch):.1f}%)")

# Among high-frequency Jeopardy concepts
for threshold in [3, 5, 10, 20]:
    subset = jeopardy[jeopardy["frequency"] >= threshold]
    n_match = subset["in_hirsch"].sum()
    print(f"  freq >= {threshold:>2}: {n_match:,} / {len(subset):,} ({100*n_match/len(subset):.1f}%)")

## 2. Coverage by Hirsch Section

In [ ]:
# Which Hirsch sections are best/worst represented on Jeopardy?
section_coverage = hirsch.groupby("section").agg(
    total=("term", "size"),
    matched=("in_jeopardy", "sum"),
).reset_index()
section_coverage["pct"] = 100 * section_coverage["matched"] / section_coverage["total"]
section_coverage = section_coverage.sort_values("pct", ascending=False)

print("Hirsch section coverage on Jeopardy:")
print(section_coverage.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
section_coverage_sorted = section_coverage.sort_values("pct")
bars = ax.barh(section_coverage_sorted["section"], section_coverage_sorted["pct"])
ax.set_xlabel("% of Hirsch entries found in Jeopardy")
ax.set_title("Jeopardy Coverage of Hirsch's Cultural Literacy Dictionary by Section")
for bar, pct in zip(bars, section_coverage_sorted["pct"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f"{pct:.0f}%", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("../data/hirsch_section_coverage.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Top Hirsch Terms Missing from Jeopardy

In [ ]:
# Hirsch terms NOT found on Jeopardy
missing_from_jeopardy = hirsch[~hirsch["in_jeopardy"]].copy()
print(f"Hirsch terms not found on Jeopardy: {len(missing_from_jeopardy):,}")
print(f"\nBy section:")
missing_by_section = missing_from_jeopardy["section"].value_counts()
for section in missing_by_section.index[:10]:
    total = len(hirsch[hirsch["section"] == section])
    missing = missing_by_section[section]
    print(f"  {section:45s} {missing:>4} / {total:>4} missing ({100*missing/total:.0f}%)")

print(f"\nSample missing terms by section:")
for section in ["World History to 1550", "Literature in English", "Fine Arts", "American Politics"]:
    subset = missing_from_jeopardy[missing_from_jeopardy["section"] == section]
    print(f"\n  {section}:")
    for _, row in subset.head(15).iterrows():
        print(f"    {row['term']}")

## 4. Top Jeopardy Concepts NOT in Hirsch

High-importance Jeopardy concepts that Hirsch didn't include — these represent cultural knowledge that emerged from media/popular culture rather than the traditional canon.

In [ ]:
# Top Jeopardy concepts by importance that are NOT in Hirsch
not_in_hirsch = jeopardy[~jeopardy["in_hirsch"]].copy()
not_in_hirsch_top = not_in_hirsch[not_in_hirsch["frequency"] >= 10].sort_values("importance", ascending=False)

print(f"Jeopardy concepts (freq >= 10) NOT in Hirsch: {len(not_in_hirsch_top):,}")
print(f"\nTop 50 by importance:")
print(not_in_hirsch_top.head(50)[["concept", "display_name", "entity_type", "frequency", 
                                    "seasons_appeared", "importance"]].to_string())

In [ ]:
# Coverage by entity type
type_coverage = jeopardy[jeopardy["frequency"] >= 5].groupby("entity_type").agg(
    total=("concept", "size"),
    in_hirsch=("in_hirsch", "sum"),
).reset_index()
type_coverage["pct"] = 100 * type_coverage["in_hirsch"] / type_coverage["total"]
type_coverage = type_coverage.sort_values("pct", ascending=False)

print("Coverage of Jeopardy concepts (freq >= 5) by entity type:")
print(type_coverage.to_string(index=False))

## 5. Overlap Summary

In [ ]:
# Final summary
print("=" * 60)
print("HIRSCH vs. JEOPARDY COMPARISON SUMMARY")
print("=" * 60)
print(f"\nHirsch Dictionary entries: {len(hirsch):,}")
print(f"Jeopardy v3 concepts: {len(jeopardy):,}")
print(f"\nHirsch entries matched in Jeopardy: {n_hirsch_in_jeopardy:,} ({100*n_hirsch_in_jeopardy/len(hirsch):.1f}%)")
print(f"Hirsch entries NOT in Jeopardy: {len(hirsch) - n_hirsch_in_jeopardy:,} ({100*(len(hirsch)-n_hirsch_in_jeopardy)/len(hirsch):.1f}%)")
print(f"\nJeopardy concepts in Hirsch (all): {n_jeopardy_in_hirsch:,} ({100*n_jeopardy_in_hirsch/len(jeopardy):.1f}%)")

for threshold in [5, 10, 20, 50]:
    subset = jeopardy[jeopardy["frequency"] >= threshold]
    n = subset["in_hirsch"].sum()
    print(f"Jeopardy concepts in Hirsch (freq >= {threshold:>2}): {n:,} / {len(subset):,} ({100*n/len(subset):.1f}%)")

print(f"\nBest-covered Hirsch sections:")
top3 = section_coverage.head(3)
for _, row in top3.iterrows():
    print(f"  {row['section']:45s} {row['pct']:.0f}%")

print(f"\nWorst-covered Hirsch sections:")
bottom3 = section_coverage.tail(3)
for _, row in bottom3.iterrows():
    print(f"  {row['section']:45s} {row['pct']:.0f}%")